# Product Wheel Schedule Optimization

This notebook solves a **mixed-integer program (MIP)** for each production line to find
the cost-optimal sequencing of SKUs across discrete time slots (the "product wheel").

### Optimization overview

| Component | Description |
|-----------|-------------|
| **Solver** | NVIDIA cuOpt (GPU-accelerated MIP) with PuLP/CBC fallback |
| **Decision variables** | Binary slot assignment (`y`), continuous production quantity (`q`), inventory (`inv`), backorder (`bo`), changeover indicator (`z`) |
| **Objective** | Minimize total changeover cost + inventory holding cost + backorder penalty |
| **Key constraints** | One product per slot, capacity/throughput limits, inventory balance, changeover linking |

### Data flow
```
ATOMIC tables ──▶ MIP solver (per line) ──▶ DATA_MART tables
  DIM_PLANT                                   FACT_LINE_SCHEDULE_OPTIMIZED
  DIM_PRODUCTION_LINE                         FACT_SERVICE_AND_SCHEDULE_KPI
  DIM_PRODUCT                                 FACT_DEMAND_FORECAST_ENRICHED
  FACT_DEMAND_FORECAST
  FACT_LINE_CALENDAR
  FACT_LINE_PRODUCT_THROUGHPUT
  FACT_LINE_PRODUCT_CHANGEOVER
  FACT_INVENTORY_POSITION
  FACT_PRODUCT_COSTING
```

## 1. Install Solver Packages

PuLP is the Python modelling interface for formulating the MIP.
cuOpt is NVIDIA's GPU-accelerated solver that PuLP can delegate to.
If cuOpt is unavailable the notebook falls back to the open-source CBC solver.

In [ ]:
import subprocess, sys

# PuLP: modelling library that translates our Python formulation into
# standard MIP formats (MPS/LP) understood by any solver backend.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pulp==2.9.0'])

# cuOpt: NVIDIA GPU-accelerated MIP solver.  Optional — CBC is the fallback.
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        '--extra-index-url=https://pypi.nvidia.com', 'cuopt-cu12'])
    print('cuOpt installed successfully')
except Exception as e:
    print(f'cuOpt install skipped (CBC fallback will be used): {e}')

In [ ]:
import pulp
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from snowflake.snowpark.context import get_active_session

# Detect whether the GPU solver is available in the runtime.
# PuLP acts as a unified API — we swap the solver backend without
# changing the model formulation.
CUOPT_AVAILABLE = False
try:
    from pulp import CUOPT
    CUOPT_AVAILABLE = True
    print('cuOpt GPU solver available')
except ImportError:
    print('cuOpt not available — will use CBC fallback')

session = get_active_session()
print(f'Snowpark session ready: {session.get_current_database()}')

## 2. Load Source Data

Pull dimension and fact tables from the ATOMIC schema into Pandas DataFrames.
These provide the sets and parameters the MIP needs:

- **DIM tables** → define the sets (plants, lines, products, formulations, customers)
- **FACT tables** → provide parameters (demand, capacity, throughput rates, changeover costs, inventory, unit costs)

In [ ]:
DB = 'PRODUCT_WHEEL_OPT'
SCHEMA = 'ATOMIC'

def read_table(table_name):
    return session.table(f'{DB}.{SCHEMA}.{table_name}').to_pandas()

df_plant = read_table('DIM_PLANT')
df_line = read_table('DIM_PRODUCTION_LINE')
df_product = read_table('DIM_PRODUCT')
df_formulation = read_table('DIM_FORMULATION')
df_customer = read_table('DIM_CUSTOMER')
df_demand = read_table('FACT_DEMAND_FORECAST')
df_calendar = read_table('FACT_LINE_CALENDAR')
df_throughput = read_table('FACT_LINE_PRODUCT_THROUGHPUT')
df_changeover = read_table('FACT_LINE_PRODUCT_CHANGEOVER')
df_inventory = read_table('FACT_INVENTORY_POSITION')
df_costing = read_table('FACT_PRODUCT_COSTING')
df_contract = read_table('FACT_CONTRACT')
df_contract_item = read_table('FACT_CONTRACT_ITEM')

print(f'Loaded: {len(df_product)} products, {len(df_line)} lines, '
      f'{len(df_demand)} demand rows, {len(df_calendar)} calendar slots, '
      f'{len(df_changeover)} changeover pairs')

## 3. Planning Horizon & Solver Settings

Filter the line calendar to the first `HORIZON_DAYS` of available slots.
Each slot corresponds to one 8-hour shift on a production line.

In [ ]:
HORIZON_DAYS = 14           # 2-week rolling window
SHIFTS_PER_DAY = 3          # 3 x 8-hour shifts
TIME_LIMIT = 120            # MIP solver time limit in seconds per line

df_calendar_avail = df_calendar[
    (df_calendar['CALENDAR_STATUS'] == 'available')
].copy()
df_calendar_avail['SLOT_DATE'] = pd.to_datetime(df_calendar_avail['TIME_SLOT_START']).dt.date
min_date = df_calendar_avail['SLOT_DATE'].min()
horizon_end = min_date + timedelta(days=HORIZON_DAYS)
df_calendar_avail = df_calendar_avail[df_calendar_avail['SLOT_DATE'] < horizon_end]

print(f'Planning horizon: {min_date} to {horizon_end}')
print(f'Available calendar slots in horizon: {len(df_calendar_avail)}')

## 4. Cost Parameters & Starting Inventory

Build the per-product cost vectors the objective function needs:

- **Inventory holding cost** — 2 % of unit cost per slot (penalises excess stock)
- **Backorder penalty** — 5× the margin per unit (heavy penalty for missed demand)
- **Starting inventory** — latest on-hand snapshot, split evenly across lines

In [ ]:
# Per-product average costs from FACT_PRODUCT_COSTING
avg_cost = df_costing.groupby('PRODUCT_ID').agg(
    material=('MATERIAL_COST_PER_UNIT', 'mean'),
    conversion=('CONVERSION_COST_PER_UNIT', 'mean'),
    margin=('STANDARD_MARGIN_PER_UNIT', 'mean')
).reset_index()

# Holding cost: small fraction of unit cost encourages just-in-time production
avg_cost['inv_holding_cost'] = (avg_cost['material'] + avg_cost['conversion']) * 0.02

# Backorder penalty: large multiplier on margin ensures demand satisfaction
# is strongly preferred over carrying inventory
avg_cost['backorder_cost'] = avg_cost['margin'] * 5.0

# Latest inventory snapshot per product (summed across plants)
latest_snap = df_inventory.groupby(['PRODUCT_ID', 'PLANT_ID'])['SNAPSHOT_TIMESTAMP'].max().reset_index()
latest_inv = pd.merge(latest_snap, df_inventory, on=['PRODUCT_ID', 'PLANT_ID', 'SNAPSHOT_TIMESTAMP'])
inv_by_product = latest_inv.groupby('PRODUCT_ID')['ON_HAND_QTY'].sum().reset_index()
inv_dict = dict(zip(inv_by_product['PRODUCT_ID'], inv_by_product['ON_HAND_QTY']))

print(f'Cost parameters computed for {len(avg_cost)} products')
print(f'Starting inventory for {len(inv_dict)} products')

## 5. MIP Formulation — `solve_line()`

This function builds and solves the product-wheel MIP **for a single production line**.
The outer loop (next cell) calls it once per line.

### Decision variables

| Variable | Type | Meaning |
|----------|------|---------|
| `y[p][t]` | Binary | 1 if product *p* is assigned to time slot *t* |
| `q[p][t]` | Continuous ≥ 0 | Quantity of product *p* produced in slot *t* |
| `inv[p][t]` | Continuous ≥ 0 | Ending inventory of product *p* after slot *t* |
| `bo[p][t]` | Continuous ≥ 0 | Backorder (unmet demand) for product *p* after slot *t* |
| `z[p][q][t]` | Binary | 1 if the line switches from product *p* (slot *t-1*) to product *q* (slot *t*) |

### Objective (minimise)
```
Σ changeover_cost(p,q) · z[p][q][t]     — penalise product switches
+ Σ inv_holding_cost(p) · inv[p][t]       — penalise excess stock
+ Σ backorder_cost(p)   · bo[p][t]        — penalise missed demand
```

### Constraints
1. **One product per slot** — `Σ_p y[p][t] ≤ 1`
2. **Capacity** — total production hours in a slot ≤ available hours
3. **Inventory balance** — `inv[t-1] + production = demand + inv[t] - bo[t]`
4. **Changeover linking** — `z[p][q][t]` can only be 1 if `y[p][t-1]=1` and `y[q][t]=1`
5. **Demand satisfaction** — backorders forced to zero at horizon end

In [ ]:
def solve_line(line_id, line_code, df_cal_line, df_tp_line, df_co_line,
               df_demand_agg, cost_df, inv0_dict):
    """
    Build and solve the product-wheel MIP for one production line.

    Returns
    -------
    schedule_rows : list[dict] or None
        One dict per assigned slot with quantities, inventory, changeover info.
    kpi : dict or None
        Aggregate KPIs for the line (fill rate, changeover hours, objective cost).
    """

    # -- Build the index sets ------------------------------------------------
    slots = sorted(df_cal_line['LINE_CALENDAR_ID'].unique())
    if len(slots) == 0:
        return None, None
    products_on_line = sorted(df_tp_line['PRODUCT_ID'].unique())
    if len(products_on_line) < 2:
        return None, None

    n_slots = min(len(slots), HORIZON_DAYS * SHIFTS_PER_DAY)
    slots = slots[:n_slots]
    products_on_line = products_on_line[:min(len(products_on_line), 15)]

    # -- Extract parameters from DataFrames ----------------------------------
    slot_cap, slot_times = {}, {}
    for _, row in df_cal_line[df_cal_line['LINE_CALENDAR_ID'].isin(slots)].iterrows():
        sid = row['LINE_CALENDAR_ID']
        slot_cap[sid] = float(row['AVAILABLE_HOURS'])
        slot_times[sid] = (row['TIME_SLOT_START'], row['TIME_SLOT_END'])

    rate = {}
    for _, row in df_tp_line[df_tp_line['PRODUCT_ID'].isin(products_on_line)].iterrows():
        rate[row['PRODUCT_ID']] = max(float(row['RUN_RATE_UNITS_PER_HOUR']), 1.0)

    change_cost = {}
    for _, row in df_co_line[
        (df_co_line['FROM_PRODUCT_ID'].isin(products_on_line)) &
        (df_co_line['TO_PRODUCT_ID'].isin(products_on_line))
    ].iterrows():
        change_cost[(row['FROM_PRODUCT_ID'], row['TO_PRODUCT_ID'])] = float(row['CHANGEOVER_COST'])

    demand_per_slot = {}
    for p in products_on_line:
        d_rows = df_demand_agg[df_demand_agg['PRODUCT_ID'] == p]
        total_demand = float(d_rows['FORECAST_QTY'].sum()) if len(d_rows) > 0 else 0
        per_slot = total_demand / max(n_slots, 1)
        for s in slots:
            demand_per_slot[(p, s)] = per_slot

    cost_map = dict(zip(cost_df['PRODUCT_ID'], cost_df.to_dict('records')))
    inv_cost, bo_cost = {}, {}
    for p in products_on_line:
        if p in cost_map:
            inv_cost[p] = cost_map[p]['inv_holding_cost']
            bo_cost[p] = cost_map[p]['backorder_cost']
        else:
            inv_cost[p] = 0.05
            bo_cost[p] = 5.0

    # Starting inventory: split total on-hand evenly across lines, cap at
    # 1 slot of demand to avoid trivially satisfying everything from stock.
    for p in products_on_line:
        raw_inv = inv0_dict.get(p, 0.0) / max(1, len(df_line))
    inv0 = {p: min(inv0_dict.get(p, 0.0) / max(1, len(df_line)),
                   demand_per_slot.get((p, slots[0]), 0) * 2)
            for p in products_on_line}

    # -- Create the PuLP model -----------------------------------------------
    model = pulp.LpProblem(f'ProductWheel_Line_{line_id}', pulp.LpMinimize)

    # Decision variables
    q_v   = pulp.LpVariable.dicts('q',   (products_on_line, slots), lowBound=0)
    y_v   = pulp.LpVariable.dicts('y',   (products_on_line, slots), cat='Binary')
    inv_v = pulp.LpVariable.dicts('inv', (products_on_line, slots), lowBound=0)
    bo_v  = pulp.LpVariable.dicts('bo',  (products_on_line, slots), lowBound=0)

    internal_slots = slots[1:]
    z_v = pulp.LpVariable.dicts('z', (products_on_line, products_on_line, internal_slots), cat='Binary')

    # Big-M: maximum quantity producible in one slot
    BIG_M = {p: rate.get(p, 100.0) * 8.0 for p in products_on_line}

    # -- Objective function ---------------------------------------------------
    changeover_expr = pulp.lpSum(
        change_cost.get((p, qq), 200.0) * z_v[p][qq][t]
        for p in products_on_line for qq in products_on_line
        for t in internal_slots if p != qq
    )
    inv_expr = pulp.lpSum(
        inv_cost[p] * inv_v[p][t]
        for p in products_on_line for t in slots
    )
    bo_expr = pulp.lpSum(
        bo_cost[p] * bo_v[p][t]
        for p in products_on_line for t in slots
    )
    model += changeover_expr + inv_expr + bo_expr

    # -- Constraint 1: one product per slot ----------------------------------
    for t in slots:
        model += pulp.lpSum(y_v[p][t] for p in products_on_line) <= 1

    # -- Constraint 2: capacity per slot -------------------------------------
    # NOTE: PuLP does not support LpVariable / float; multiply by reciprocal.
    for t in slots:
        model += pulp.lpSum(
            (1.0 / rate.get(p, 100.0)) * q_v[p][t]
            for p in products_on_line
        ) <= slot_cap.get(t, 8.0)

    # -- Constraint 2b: big-M linking (produce only when assigned) -----------
    # Without this, the solver can set q>0 with y=0, decoupling assignment
    # from production.
    for p in products_on_line:
        for t in slots:
            model += q_v[p][t] <= BIG_M[p] * y_v[p][t]

    # -- Constraint 3: inventory balance -------------------------------------
    # prev_inventory + production = demand + ending_inventory - backorder
    for p in products_on_line:
        for idx, t in enumerate(slots):
            prev = inv0[p] if idx == 0 else inv_v[p][slots[idx - 1]]
            model += prev + q_v[p][t] == demand_per_slot.get((p, t), 0) + inv_v[p][t] - bo_v[p][t]

    # -- Constraint 4: changeover linking ------------------------------------
    # z[p][q][t] = 1 only if product p ran in t-1 AND product q runs in t.
    for idx, t in enumerate(slots):
        if idx == 0:
            continue
        prev_t = slots[idx - 1]
        for p in products_on_line:
            for qq in products_on_line:
                if p != qq:
                    model += z_v[p][qq][t] <= y_v[p][prev_t]
                    model += z_v[p][qq][t] <= y_v[qq][t]
        model += pulp.lpSum(
            z_v[p][qq][t] for p in products_on_line
            for qq in products_on_line if p != qq
        ) <= pulp.lpSum(y_v[qq][t] for qq in products_on_line)

    # -- Constraint 5: no backorders at horizon end --------------------------
    for p in products_on_line:
        model += bo_v[p][slots[-1]] == 0

    # -- Select and run the solver -------------------------------------------
    if CUOPT_AVAILABLE:
        try:
            solver = CUOPT(msg=0, timeLimit=TIME_LIMIT)
        except Exception:
            solver = pulp.PULP_CBC_CMD(timeLimit=TIME_LIMIT, msg=0, threads=8)
    else:
        solver = pulp.PULP_CBC_CMD(timeLimit=TIME_LIMIT, msg=0, threads=8)

    model.solve(solver)
    status = pulp.LpStatus[model.status]
    obj_val = pulp.value(model.objective) if model.status == 1 else None
    print(f'  Line {line_code}: status={status}, objective={obj_val}')

    if model.status != 1:
        return None, None

    # -- Extract solution ----------------------------------------------------
    schedule_rows = []
    for t in slots:
        for p in products_on_line:
            yval = y_v[p][t].varValue
            if yval is not None and yval > 0.5:
                qty_val = q_v[p][t].varValue or 0
                inv_out = inv_v[p][t].varValue or 0
                ts_start, ts_end = slot_times.get(t, (None, None))
                co_time = 0
                if t in internal_slots:
                    for pp in products_on_line:
                        if pp != p:
                            zval = z_v[pp][p][t].varValue
                            if zval is not None and zval > 0.5:
                                co_time = change_cost.get((pp, p), 0) / 200.0
                schedule_rows.append({
                    'LINE_ID': line_id,
                    'TIME_SLOT_START': ts_start,
                    'TIME_SLOT_END': ts_end,
                    'PRODUCT_ID': int(p),
                    'PLANNED_QTY': round(qty_val, 1),
                    'PROJECTED_INVENTORY_QTY': round(inv_out, 1),
                    'TOTAL_CHANGEOVER_TIME_HOURS': round(co_time, 2),
                    'OBJECTIVE_COST': round(obj_val, 2) if obj_val else 0
                })

    total_demand = sum(demand_per_slot.get((p, t), 0) for p in products_on_line for t in slots)
    total_produced = sum(r['PLANNED_QTY'] for r in schedule_rows)
    fill_rate = min(total_produced / max(total_demand, 1), 1.0)
    for r in schedule_rows:
        r['PROJECTED_FILL_RATE'] = round(fill_rate, 4)

    print(f'    -> {len(schedule_rows)} slots assigned, {total_produced:.0f} units planned vs {total_demand:.0f} demanded')

    kpi = {
        'LINE_ID': line_id,
        'LINE_CODE': line_code,
        'FILL_RATE': fill_rate,
        'TOTAL_CHANGEOVER_HOURS': sum(r['TOTAL_CHANGEOVER_TIME_HOURS'] for r in schedule_rows),
        'TOTAL_PLANNED_QTY': total_produced,
        'TOTAL_DEMAND_QTY': total_demand,
        'OBJECTIVE_COST': obj_val
    }
    return schedule_rows, kpi

## 6. Solve All Production Lines

Iterate over every line in `DIM_PRODUCTION_LINE`, filter the relevant
calendar / throughput / changeover data, and call `solve_line()`.

In [ ]:
scenario_id = f'OPT-{datetime.now().strftime("%Y%m%d-%H%M%S")}'
print(f'Scenario: {scenario_id}')

all_schedule_rows = []
all_kpi_rows = []
demand_agg = df_demand.groupby('PRODUCT_ID')['FORECAST_QTY'].sum().reset_index()

for _, line_row in df_line.iterrows():
    lid = line_row['LINE_ID']
    lcode = line_row['LINE_CODE']
    plant_id = line_row['PLANT_ID']

    cal_line = df_calendar_avail[df_calendar_avail['LINE_ID'] == lid]
    tp_line = df_throughput[df_throughput['LINE_ID'] == lid]
    co_line = df_changeover[df_changeover['LINE_ID'] == lid]

    print(f'Solving line {lcode} ({len(cal_line)} slots, {len(tp_line)} products)...')
    sched, kpi = solve_line(lid, lcode, cal_line, tp_line, co_line,
                            demand_agg, avg_cost, inv_dict)
    if sched:
        for r in sched:
            r['SCENARIO_ID'] = scenario_id
        all_schedule_rows.extend(sched)
    if kpi:
        plant_row = df_plant[df_plant['PLANT_ID'] == plant_id].iloc[0]
        kpi['SCENARIO_ID'] = scenario_id
        kpi['PLANT_NAME'] = plant_row['PLANT_NAME']
        all_kpi_rows.append(kpi)

print(f'\nTotal schedule rows: {len(all_schedule_rows)}')
print(f'Lines solved: {len(all_kpi_rows)}')

## 7. Write Optimized Schedule to DATA_MART

Persist the slot-level schedule to `FACT_LINE_SCHEDULE_OPTIMIZED`.
This is the primary table consumed by Streamlit to render the Gantt chart
and schedule drill-down.

In [ ]:
if all_schedule_rows:
    sched_df = pd.DataFrame(all_schedule_rows)

    # Align column order to match the target table schema exactly.
    sched_df = sched_df[[
        'SCENARIO_ID', 'LINE_ID', 'TIME_SLOT_START', 'TIME_SLOT_END',
        'PRODUCT_ID', 'PLANNED_QTY', 'PROJECTED_FILL_RATE',
        'PROJECTED_INVENTORY_QTY', 'TOTAL_CHANGEOVER_TIME_HOURS',
        'OBJECTIVE_COST'
    ]]

    # Truncate then append via create_dataframe for type safety.
    session.sql('TRUNCATE TABLE PRODUCT_WHEEL_OPT.DATA_MART.FACT_LINE_SCHEDULE_OPTIMIZED').collect()
    sp_sched = session.create_dataframe(sched_df)
    for col in sched_df.columns:
        sp_sched = sp_sched.with_column_renamed(col, col.upper())
    sp_sched.write.mode('append').save_as_table(
        'PRODUCT_WHEEL_OPT.DATA_MART.FACT_LINE_SCHEDULE_OPTIMIZED'
    )
    print(f'Wrote {len(sched_df)} rows to FACT_LINE_SCHEDULE_OPTIMIZED')
else:
    print('No schedule rows to write')

## 8. Write Service & Schedule KPIs

Aggregate fill rate, changeover hours, and planned vs. demand quantities
per line × product family and persist to `FACT_SERVICE_AND_SCHEDULE_KPI`.
Streamlit uses this for the KPI scorecard and trend charts.

In [ ]:
if all_kpi_rows:
    kpi_records = []
    for kpi in all_kpi_rows:
        for fam in df_product['PRODUCT_FAMILY'].unique():
            kpi_records.append({
                'SCENARIO_ID': kpi['SCENARIO_ID'],
                'PLANT_NAME': kpi['PLANT_NAME'],
                'LINE_CODE': kpi['LINE_CODE'],
                'PRODUCT_FAMILY': fam,
                'CUSTOMER_NAME': 'All Customers',
                'WEEK_START': min_date.isoformat(),
                'FILL_RATE': round(kpi['FILL_RATE'], 4),
                'INVENTORY_DAYS_OF_SUPPLY': round(
                    kpi['TOTAL_PLANNED_QTY'] / max(kpi['TOTAL_DEMAND_QTY'] / max(HORIZON_DAYS, 1), 1), 1
                ),
                'CHANGEOVER_HOURS': round(kpi['TOTAL_CHANGEOVER_HOURS'], 2),
                'TOTAL_PLANNED_QTY': round(kpi['TOTAL_PLANNED_QTY'], 1),
                'TOTAL_DEMAND_QTY': round(kpi['TOTAL_DEMAND_QTY'], 1),
                'OBJECTIVE_COST': round(kpi['OBJECTIVE_COST'], 2) if kpi['OBJECTIVE_COST'] else 0
            })
    kpi_df = pd.DataFrame(kpi_records)

    session.sql('TRUNCATE TABLE PRODUCT_WHEEL_OPT.DATA_MART.FACT_SERVICE_AND_SCHEDULE_KPI').collect()

    cols = ['SCENARIO_ID', 'PLANT_NAME', 'LINE_CODE', 'PRODUCT_FAMILY',
            'CUSTOMER_NAME', 'WEEK_START', 'FILL_RATE', 'INVENTORY_DAYS_OF_SUPPLY',
            'CHANGEOVER_HOURS', 'TOTAL_PLANNED_QTY', 'TOTAL_DEMAND_QTY', 'OBJECTIVE_COST']
    col_list = ', '.join(cols)

    for _, r in kpi_df.iterrows():
        vals = []
        for c in cols:
            v = r[c]
            if c == 'WEEK_START':
                vals.append(f"TO_DATE('{v}', 'YYYY-MM-DD')")
            elif isinstance(v, str):
                safe = str(v).replace("'", "''")
                vals.append(f"'{safe}'")
            else:
                vals.append(str(v))
        val_list = ', '.join(vals)
        session.sql(f'INSERT INTO PRODUCT_WHEEL_OPT.DATA_MART.FACT_SERVICE_AND_SCHEDULE_KPI ({col_list}) VALUES ({val_list})').collect()

    print(f'Wrote {len(kpi_df)} rows to FACT_SERVICE_AND_SCHEDULE_KPI (SQL INSERT, KPI_ID auto-populated)')
else:
    print('No KPI rows to write')

## 9. Write Enriched Demand Forecast

Augment the raw demand forecast with 85th / 115th percentile prediction
intervals and tag with the scenario ID. Streamlit uses this to overlay
forecast uncertainty on the demand-vs-production charts.

In [ ]:
if all_schedule_rows:
    enriched_rows = []
    for _, row in df_demand.iterrows():
        base_qty = float(row['FORECAST_QTY'])
        ws = row['FORECAST_WEEK_START']
        we = row['FORECAST_WEEK_END']
        enriched_rows.append({
            'CUSTOMER_ID': int(row['CUSTOMER_ID']),
            'PRODUCT_ID': int(row['PRODUCT_ID']),
            'WEEK_START': ws.isoformat() if hasattr(ws, 'isoformat') else str(ws),
            'WEEK_END': we.isoformat() if hasattr(we, 'isoformat') else str(we),
            'FORECAST_QTY': base_qty,
            'PREDICTION_INTERVAL_LOW': round(base_qty * 0.85, 1),
            'PREDICTION_INTERVAL_HIGH': round(base_qty * 1.15, 1),
            'MODEL_VERSION': scenario_id
        })
    enr_df = pd.DataFrame(enriched_rows)

    # Truncate then append via create_dataframe to avoid Parquet nanosecond→DATE cast errors.
    session.sql('TRUNCATE TABLE PRODUCT_WHEEL_OPT.DATA_MART.FACT_DEMAND_FORECAST_ENRICHED').collect()
    sp_enr = session.create_dataframe(enr_df)
    for col in enr_df.columns:
        sp_enr = sp_enr.with_column_renamed(col, col.upper())
    sp_enr.write.mode('append').save_as_table(
        'PRODUCT_WHEEL_OPT.DATA_MART.FACT_DEMAND_FORECAST_ENRICHED'
    )
    print(f'Wrote {len(enr_df)} rows to FACT_DEMAND_FORECAST_ENRICHED')
else:
    print('No enriched forecast rows to write')

## 10. Run Summary

In [ ]:
print('=' * 60)
print(f'Scenario: {scenario_id}')
print(f'Solver:   {"cuOpt (GPU)" if CUOPT_AVAILABLE else "CBC (CPU)"}')
print(f'Horizon:  {min_date} → {horizon_end} ({HORIZON_DAYS} days)')
print(f'Lines solved: {len(all_kpi_rows)}')
print(f'Schedule rows written: {len(all_schedule_rows)}')
print('-' * 60)

for kpi in all_kpi_rows:
    print(f"  {kpi['PLANT_NAME']:20s} / {kpi['LINE_CODE']:12s}  "
          f"fill_rate={kpi['FILL_RATE']:.1%}  "
          f"changeover_hrs={kpi['TOTAL_CHANGEOVER_HOURS']:.1f}  "
          f"objective=${kpi['OBJECTIVE_COST']:,.0f}")

print('=' * 60)
print('DATA_MART tables ready for Streamlit:')
print('  • FACT_LINE_SCHEDULE_OPTIMIZED   — slot-level Gantt data')
print('  • FACT_SERVICE_AND_SCHEDULE_KPI   — scorecard metrics')
print('  • FACT_DEMAND_FORECAST_ENRICHED   — forecast + intervals')